# siann's TPU probe

Run on a TPU (Colab: Runtime, Change runtime type, TPU). Two parts, each in its own process:

1. **JAX:** exact search through JAX's built-in operations (L2 and inner product by a matrix multiply and
   `lax.top_k` or the TPU's `lax.approx_min_k`, at the default matrix-multiply precision and at full float32;
   L1 by tiles), and L1 through siann's custom kernel in Pallas, checked first against exact search.
2. **PyTorch:** the same searches through PyTorch's built-in operations on TorchTPU (or PyTorch/XLA), eagerly
   and compiled.

Every result is checked against NumPy, and each number prints as it is measured. Copy the outputs of the check
and benchmark cells back.

In [ ]:
# which TPU this runtime has, and Python's version (TorchTPU needs 3.11 or later)
import os
import sys

print(sys.version)
for var in ("TPU_ACCELERATOR_TYPE", "TPU_NAME", "COLAB_TPU_ADDR"):
    print(var, os.environ.get(var))
print("accelerator devices:", sorted(f for f in os.listdir("/dev") if f.startswith(("accel", "vfio"))))

## 1. JAX

In [ ]:
# JAX for TPUs (Colab's TPU runtimes come with it; reinstalled only if it cannot see the TPU)
import subprocess
import sys

r = subprocess.run([sys.executable, "-c", "import jax; print(jax.__version__, jax.devices())"], check=False,
                   capture_output=True, text=True)
if "Tpu" not in r.stdout and "TPU" not in r.stdout:
    print("JAX cannot see a TPU:", (r.stdout + r.stderr).strip().splitlines()[-1:] or "?")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[tpu]"], check=False)
    r = subprocess.run([sys.executable, "-c", "import jax; print(jax.__version__, jax.devices())"], check=False,
                       capture_output=True, text=True)
print((r.stdout + r.stderr).strip())

In [ ]:
%%writefile jax_probe.py
"""A probe of exact search on a Google TPU through JAX: built-in operations for every metric, and a custom kernel
for L1 in Pallas (JAX's kernel language, which compiles for TPUs and NVIDIA GPUs, and which PyTorch reaches on
TPUs through TorchTPU).

Built-in: L2 and inner product as a matrix multiply followed by lax.top_k, or by lax.approx_min_k (the TPU's own
approximate top-k, at a recall target of 0.95), at the TPU's default matrix-multiply precision (bfloat16 inputs)
and at full float32 (HIGHEST); L1 as a broadcast difference summed over the coordinates, a chunk of the base at a
time, with the best k kept across chunks (lax.scan).

Pallas, L1: a grid of (query block, base block) programs; each computes its tile of L1 distances in the TPU's
on-chip memory and merges it into its query block's best k, kept in the output across the base blocks, by k
passes of a minimum (ties by index, as siann's other kernels), so no distance matrix ever reaches memory.

Every result is checked against exact search in NumPy on a sample of queries (recall@k and the share of queries
whose neighbors are exactly NumPy's). Memory: the device's bytes in use and its peak so far, where it reports them.

    python jax_probe.py                                   # on a TPU (Colab: Runtime, Change runtime type, TPU)
    python jax_probe.py --check                           # the Pallas kernel against NumPy, small shapes
    python jax_probe.py --check --interpret               # the same anywhere, through Pallas's TPU interpreter
"""
import argparse
import functools
import time

import jax
import jax.numpy as jnp
import numpy as np
from jax import lax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

CHUNK = 256                     # queries per matrix multiply (256 x 1,000,000 float32 scores is 1 GB)
L1_CHUNK = 16_384               # base vectors per step of the built-in L1 scan
KP = 128                        # the kept best-k, padded to the TPU's 128 lanes
BIG = 3.0e38                    # "no candidate" (finite: inf - inf would be NaN in a padded row)
PAD = 1.0e30                    # padding rows of the base: never nearer than a real row


def exact_numpy(xb, xq, k, metric):
    """NumPy's k nearest (int64 indices, ties by index): the reference."""
    out = []
    for q in xq:
        if metric == "l2":
            d = ((xb - q) ** 2).sum(axis=1)
        elif metric == "ip":
            d = -(xb @ q)
        else:
            d = np.abs(xb - q).sum(axis=1)
        part = np.argpartition(d, k)[:k + 1] if len(d) > k + 1 else np.arange(len(d))
        cand = np.flatnonzero(d <= d[part].max())
        out.append(cand[np.lexsort((cand, d[cand]))][:k])
    return np.array(out)


# --- Pallas: L1 with the selection fused -------------------------------------------------------------------------

def _l1_kernel(q_ref, x_ref, v_ref, i_ref, *, k, block_n):
    j = pl.program_id(1)

    @pl.when(j == 0)
    def _():
        v_ref[...] = jnp.full(v_ref.shape, BIG, jnp.float32)
        i_ref[...] = jnp.full(i_ref.shape, jnp.iinfo(jnp.int32).max, jnp.int32)

    q, x = q_ref[...], x_ref[...]
    d = jnp.sum(jnp.abs(q[:, None, :] - x[None, :, :]), axis=2)              # (BQ, BN), on chip
    di = j * block_n + lax.broadcasted_iota(jnp.int32, d.shape, 1)
    kv, ki = v_ref[...], i_ref[...]                                            # (BQ, KP), the best so far
    lane = lax.broadcasted_iota(jnp.int32, kv.shape, 1)
    nv = jnp.full(kv.shape, BIG, jnp.float32)
    ni = jnp.full(ki.shape, jnp.iinfo(jnp.int32).max, jnp.int32)
    for p in range(k):                                                         # k passes of a minimum
        m = jnp.minimum(jnp.min(kv, axis=1, keepdims=True), jnp.min(d, axis=1, keepdims=True))
        top = jnp.iinfo(jnp.int32).max
        mi = jnp.minimum(jnp.min(jnp.where(kv == m, ki, top), axis=1, keepdims=True),
                         jnp.min(jnp.where(d == m, di, top), axis=1, keepdims=True))   # ties: the smaller index
        nv = jnp.where(lane == p, m, nv)
        ni = jnp.where(lane == p, mi, ni)
        kv = jnp.where((kv == m) & (ki == mi), BIG, kv)
        d = jnp.where((d == m) & (di == mi), BIG, d)
    v_ref[...] = nv
    i_ref[...] = ni


@functools.partial(jax.jit, static_argnames=("k", "block_q", "block_n", "interpret"))
def pallas_l1(x, q, k, block_q=8, block_n=512, interpret=False):
    """Each query's k nearest rows of x by L1 (int32 indices, nearest first, ties by index), through the kernel.
    x must already be padded to a multiple of block_n (pad_base)."""
    nq, d = q.shape
    n = x.shape[0]
    mq = -(-nq // block_q) * block_q
    q = jnp.pad(q, ((0, mq - nq), (0, 0)))
    grid = (mq // block_q, n // block_n)
    params = pltpu.CompilerParams(dimension_semantics=("parallel", "arbitrary"))
    _, i = pl.pallas_call(
        functools.partial(_l1_kernel, k=k, block_n=block_n),
        grid=grid,
        in_specs=[pl.BlockSpec((block_q, d), lambda a, b: (a, 0)),
                  pl.BlockSpec((block_n, d), lambda a, b: (b, 0))],
        out_specs=[pl.BlockSpec((block_q, KP), lambda a, b: (a, 0)),
                   pl.BlockSpec((block_q, KP), lambda a, b: (a, 0))],
        out_shape=[jax.ShapeDtypeStruct((mq, KP), jnp.float32), jax.ShapeDtypeStruct((mq, KP), jnp.int32)],
        compiler_params=params,
        interpret=pltpu.InterpretParams() if interpret else False,
    )(q, x)
    return i[:nq, :k]


def pad_base(x, block_n):
    n = x.shape[0]
    m = -(-n // block_n) * block_n
    return jnp.pad(x, ((0, m - n), (0, 0)), constant_values=PAD)


# --- built-in operations -----------------------------------------------------------------------------------------

def builtin_searches(x, k):
    sq = jnp.sum(x * x, axis=1)

    def scores(q, metric, precision):
        s = jnp.matmul(q, x.T, precision=precision)
        return sq[None, :] - 2 * s if metric == "l2" else -s

    def exact(q, metric, precision):
        return lax.top_k(-scores(q, metric, precision), k)[1]

    def approx(q, metric, precision):
        return lax.approx_min_k(scores(q, metric, precision), k, recall_target=0.95)[1]

    out = {}
    for metric in ("l2", "ip"):
        for pname, precision in (("default", lax.Precision.DEFAULT), ("highest", lax.Precision.HIGHEST)):
            out[(metric, f"matmul + top_k, precision {pname}")] = jax.jit(
                functools.partial(exact, metric=metric, precision=precision))
            out[(metric, f"matmul + approx_min_k, precision {pname}")] = jax.jit(
                functools.partial(approx, metric=metric, precision=precision))
    return out


def builtin_l1(x, k):
    n, d = x.shape
    m = -(-n // L1_CHUNK) * L1_CHUNK
    xs = jnp.pad(x, ((0, m - n), (0, 0)), constant_values=PAD).reshape(m // L1_CHUNK, L1_CHUNK, d)

    @jax.jit
    def search(q):
        def step(carry, t):
            bv, bi = carry
            xc, base = t
            dist = jnp.sum(jnp.abs(q[:, None, :] - xc[None, :, :]), axis=2)
            v, i = lax.top_k(-dist, k)
            v, i = jnp.concatenate([bv, -v], 1), jnp.concatenate([bi, i + base], 1)
            top = lax.top_k(-v, k)[1]
            return (jnp.take_along_axis(v, top, 1), jnp.take_along_axis(i, top, 1)), None
        init = (jnp.full((q.shape[0], k), BIG, jnp.float32), jnp.zeros((q.shape[0], k), jnp.int32))
        (_, bi), _ = lax.scan(step, init, (xs, jnp.arange(m // L1_CHUNK, dtype=jnp.int32) * L1_CHUNK))
        return bi
    return search


def run_chunks(fn, q, size):
    """fn over q in chunks of one fixed size (the last one padded), so each jitted fn compiles once."""
    out = []
    for s in range(0, q.shape[0], size):
        qs = q[s:s + size]
        if qs.shape[0] < size:
            qs = jnp.concatenate([qs, jnp.broadcast_to(qs[:1], (size - qs.shape[0], qs.shape[1]))])
        out.append(fn(qs)[:min(size, q.shape[0] - s)])
    return jnp.concatenate(out)


def memory():
    """The device's bytes in use and its peak so far, MB (None where it does not say)."""
    try:
        st = jax.devices()[0].memory_stats() or {}
    except Exception:  # noqa: BLE001
        st = {}
    get = st.get
    return tuple(None if get(key) is None else get(key) / 2**20 for key in ("bytes_in_use", "peak_bytes_in_use"))


def check(interpret: bool) -> None:
    rng = np.random.default_rng(1)
    print(f"  on {jax.devices()[0].platform}{', through the TPU interpreter' if interpret else ''}", flush=True)
    for n, d, nq, k, bq, bn in ((1000, 128, 37, 10, 8, 128), (777, 128, 5, 3, 8, 256), (2048, 128, 20, 10, 16, 512)):
        x = rng.integers(0, 4, (n, d)).astype(np.float32)              # small integers: many ties
        q = rng.integers(0, 4, (nq, d)).astype(np.float32)
        truth = exact_numpy(x, q, k, "l1")
        got = np.asarray(pallas_l1(pad_base(jnp.asarray(x), bn), jnp.asarray(q), k, bq, bn, interpret))
        same = np.array_equal(got, truth)
        print(f"  n {n}, d {d}, {nq} queries, k {k}, blocks {bq} x {bn}: identical to exact search, ties "
              f"included: {same}", flush=True)
        assert same


def bench(n: int, d: int, batches, l1_batch: int, check_queries: int) -> None:
    k = 10
    dev = jax.devices()[0]
    rng = np.random.default_rng(0)
    xb = rng.standard_normal((n, d), dtype=np.float32)
    xq = rng.standard_normal((max(batches), d), dtype=np.float32)
    print(f"JAX {jax.__version__} on {dev.device_kind} ({dev.platform}): {n:,} Gaussian vectors, {d} dimensions, "
          f"k = {k}", flush=True)
    t = time.perf_counter()
    truth = {m: exact_numpy(xb, xq[:check_queries], k, m) for m in ("l2", "ip")}
    truth["l1"] = exact_numpy(xb, xq[:min(check_queries, l1_batch)], k, "l1")
    print(f"  NumPy's exact neighbors for the check: {time.perf_counter() - t:.1f} s", flush=True)
    x, q = jnp.asarray(xb), jnp.asarray(xq)

    def timed(call, qb):
        call(qb).block_until_ready()                                   # compiles, warms up
        best = float("inf")
        for _ in range(3):
            t = time.perf_counter()
            call(qb).block_until_ready()
            best = min(best, time.perf_counter() - t)
        return qb.shape[0] / best

    def agreement(found, ref):
        found = np.asarray(found)[:len(ref)]
        recall = np.mean([len(set(f) & set(r)) / len(r) for f, r in zip(found, ref, strict=True)])
        return recall, np.mean([np.array_equal(f, r) for f, r in zip(found, ref, strict=True)])

    def mem():
        used, peak = memory()
        return f"{'?' if used is None else f'{used:,.0f}'} / {'?' if peak is None else f'{peak:,.0f}'}"

    print("  queries per second; then the device's memory after the row, MB: in use / peak so far; then recall@10 "
          "and exact matches against NumPy", flush=True)
    print(f"  {'':44s}" + "".join(f"{f'batch {b:,}':>14s}" for b in batches), flush=True)
    for (metric, how), fn in builtin_searches(x, k).items():
        def call(qb, fn=fn):
            return run_chunks(fn, qb, CHUNK)
        print(f"  {f'{metric}, {how}':44s}", end="", flush=True)
        try:
            for b in batches:
                print(f"{timed(call, q[:b]):14,.0f}", end="", flush=True)
            recall, same = agreement(call(q[:check_queries]), truth[metric])
            print(f"\n      memory {mem()}; recall@10 {recall:.4f}, exact matches {same:.4f}", flush=True)
        except Exception as e:  # noqa: BLE001 (report and go on: one failure should not hide the rest)
            print(f"\n      failed: {type(e).__name__}: {str(e)[:300]}", flush=True)

    qb = q[:l1_batch]
    print(f"\n  L1, batch {l1_batch} (queries per second)", flush=True)
    search = builtin_l1(x, k)
    try:
        def call_builtin(qq):
            return run_chunks(search, qq, 16)
        qps = timed(call_builtin, qb)
        recall, same = agreement(call_builtin(qb), truth["l1"])
        print(f"    built-in tiles (lax.scan){qps:18,.0f}   memory {mem()}; recall@10 {recall:.4f}, exact matches "
              f"{same:.4f}", flush=True)
    except Exception as e:  # noqa: BLE001
        print(f"    built-in tiles failed: {type(e).__name__}: {str(e)[:300]}", flush=True)
    for bq, bn in ((8, 512), (8, 1024), (16, 512), (32, 256), (8, 2048)):
        try:
            xp = pad_base(x, bn)

            def call_pallas(qq, xp=xp, bq=bq, bn=bn):
                return pallas_l1(xp, qq, k, bq, bn)
            qps = timed(call_pallas, qb)
            recall, same = agreement(call_pallas(qb), truth["l1"])
            print(f"    Pallas, blocks {bq:>2} x {bn:<5}{qps:18,.0f}   memory {mem()}; recall@10 {recall:.4f}, "
                  f"exact matches {same:.4f}", flush=True)
        except Exception as e:  # noqa: BLE001
            print(f"    Pallas, blocks {bq} x {bn} failed: {type(e).__name__}: {str(e)[:300]}", flush=True)


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--check", action="store_true")
    p.add_argument("--interpret", action="store_true", help="with --check: through Pallas's TPU interpreter")
    p.add_argument("--n", type=int, default=1_000_000)
    p.add_argument("--d", type=int, default=128)
    p.add_argument("--batches", default="100,1000,10000")
    p.add_argument("--l1-batch", type=int, default=1000)
    p.add_argument("--check-queries", type=int, default=100)
    a = p.parse_args()
    if a.check:
        check(a.interpret)
    else:
        bench(a.n, a.d, tuple(map(int, a.batches.split(","))), a.l1_batch, a.check_queries)


if __name__ == "__main__":
    main()

In [ ]:
!python -u jax_probe.py --check

In [ ]:
!python -u jax_probe.py

## 2. PyTorch

In [ ]:
# TorchTPU (validated on TPU v5e, v5p, v6e and v7); where it cannot run here, PyTorch/XLA if the runtime has it
import subprocess
import sys


def works(stmt):
    r = subprocess.run([sys.executable, "-c", stmt], check=False, capture_output=True, text=True)
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]


TPU = "import torch\ntry:\n    import torch_tpu\nexcept ImportError:\n    pass\ntorch.ones(1, device='tpu')"
ok, why = works(TPU)
if not ok:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil", "torch_tpu",
                        "--extra-index-url", "https://download.pytorch.org/whl/cpu"],
                       check=False, capture_output=True, text=True)
    print("torch_tpu", "installed" if r.returncode == 0 else "not installed: " + (r.stderr.strip().splitlines() or ["?"])[-1])
    ok, why = works(TPU)
print("TorchTPU", "works" if ok else f"does not work here: {why}")
if not ok:
    ok, why = works("import torch_xla; print(torch_xla.device())")
    print("PyTorch/XLA", "works: the probe uses it" if ok else f"does not work either: {why}")

In [ ]:
%%writefile tpu_probe.py
"""A probe of exact search on a Google TPU through PyTorch's built-in operations (no custom kernels: Triton has
no TPU target): L2 and inner product as a matrix multiply followed by a top-k, the top-k taken once over each
query's whole row or in two stages (the best k of each block of the base, then the best k of those, siann's
two-stage design), and L1 as a broadcast difference summed over the coordinates, kept to the best k per chunk of
the base. Each runs eagerly and through torch.compile, and the matrix multiplies at the TPU's default precision
(bfloat16 inputs, one pass) and at full float32 ("highest", six passes).

Every result is checked against exact search in NumPy on a sample of queries (recall@k, and the share of
queries whose k neighbors are exactly NumPy's). Speeds are queries per second on Gaussian vectors; memory is the
accelerator's peak where its backend reports one, and the host process's peak.

The TPU is reached through TorchTPU (``torch_tpu``, the device "tpu") where it is installed, else through
PyTorch/XLA (``torch_xla``). ``--device cpu`` runs the same code on PyTorch's CPU device, to check it anywhere.

    python tpu_probe.py                                  # on a TPU (Colab: Runtime, Change runtime type, TPU)
    python tpu_probe.py --device cpu --n 20000 --batches 100,300
"""
import argparse
import contextlib
import threading
import time

import numpy as np
import torch

CHUNK = 256          # queries per matrix multiply: 256 x 1,000,000 float32 scores is 1 GB
BLOCK = 1000         # the first stage's block of the base (n must be a multiple)
L1_Q, L1_N = 8, 31_250   # L1's tile: 8 queries x 31,250 vectors x 128 coordinates is 128 MB


class Backend:
    """The accelerator: its device, how to wait for it, its torch.compile backend and its memory counters."""

    def __init__(self, want: str):
        self.name, self.compile_backend, self._xm = "cpu", None, None
        self.device = torch.device("cpu")
        if want in ("auto", "tpu"):
            try:
                try:
                    import torch_tpu  # noqa: F401 (registers the "tpu" device where autoloading does not)
                except ImportError:
                    pass
                torch.ones(1, device="tpu")
                self.name, self.device, self.compile_backend = "torchtpu", torch.device("tpu"), "tpu"
                return
            except Exception as e:  # noqa: BLE001 (TorchTPU missing or unusable here: try PyTorch/XLA)
                print(f"  (TorchTPU not usable: {type(e).__name__}: {str(e)[:200]})", flush=True)
            try:
                import torch_xla
                import torch_xla.core.xla_model as xm
                self.name, self.device, self.compile_backend, self._xm = "torch_xla", torch_xla.device(), "openxla", xm
                return
            except Exception as e:  # noqa: BLE001
                print(f"  (PyTorch/XLA not usable: {type(e).__name__}: {str(e)[:200]})", flush=True)
            if want == "tpu":
                raise SystemExit("no TPU backend: install torch_tpu (or torch_xla) on a TPU runtime")

    def describe(self) -> str:
        if self.name == "torchtpu":
            try:
                p = torch.tpu.get_device_properties(0)
                return f"TorchTPU, {getattr(p, 'name', '?')} ({getattr(p, 'version', '?')}), " \
                       f"{getattr(p, 'hbm_bytes_per_chip', 0) / 2**30:.0f} GB HBM"
            except Exception:  # noqa: BLE001
                return "TorchTPU"
        if self.name == "torch_xla":
            try:
                import torch_xla
                return f"PyTorch/XLA {torch_xla.__version__}, {self._xm.xla_device_hw(self.device)}"
            except Exception:  # noqa: BLE001
                return "PyTorch/XLA"
        return "PyTorch's CPU device"

    def sync(self) -> None:
        if self.name == "torchtpu":
            torch.accelerator.synchronize()
        elif self.name == "torch_xla":
            import torch_xla
            torch_xla.sync()
            self._xm.wait_device_ops()

    def reset_peak(self) -> None:
        if self.name == "torchtpu":
            with contextlib.suppress(Exception):          # no peak counter: the peaks read None
                torch.accelerator.reset_peak_memory_stats()

    def peak_mb(self):
        """The accelerator's peak allocation since the last reset (TorchTPU), or since the start (PyTorch/XLA)."""
        try:
            if self.name == "torchtpu":
                return torch.accelerator.max_memory_allocated() / 2**20
            if self.name == "torch_xla":
                info = self._xm.get_memory_info(self.device)
                peak = info.get("peak_bytes_used", info.get("bytes_used"))
                return None if peak is None else peak / 2**20
        except Exception:  # noqa: BLE001
            return None
        return None


class HostPeak:
    """The host process's peak resident memory during a block, MB above its start (polled every 10 ms)."""

    def __enter__(self):
        try:
            import psutil
            self._proc = psutil.Process()
        except ImportError:
            self._proc, self.mb = None, None
            return self
        self.start = self.top = self._proc.memory_info().rss
        self._stop = threading.Event()

        def poll():
            while not self._stop.is_set():
                self.top = max(self.top, self._proc.memory_info().rss)
                time.sleep(0.01)
        self._thread = threading.Thread(target=poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        if self._proc is not None:
            self._stop.set()
            self._thread.join()
            self.mb = (max(self.top, self._proc.memory_info().rss) - self.start) / 2**20


def exact_numpy(xb, xq, k, metric):
    """NumPy's k nearest (int64 indices, ties by index) for a sample of queries: the reference."""
    out = []
    for q in xq:
        if metric == "l2":
            d = ((xb - q) ** 2).sum(axis=1)
        elif metric == "ip":
            d = -(xb @ q)
        else:
            d = np.abs(xb - q).sum(axis=1)
        part = np.argpartition(d, k)[:k + 1] if len(d) > k + 1 else np.arange(len(d))
        cand = np.flatnonzero(d <= d[part].max())
        out.append(cand[np.lexsort((cand, d[cand]))][:k])
    return np.array(out)


def make_searches(x, k):
    """Each search maps a fixed-size chunk of queries to its k nearest indices (int64, on the device)."""
    n = x.shape[0]
    sq = (x * x).sum(dim=1)

    def scores(q, metric):
        s = q @ x.T
        return sq[None, :] - 2 * s if metric == "l2" else -s

    def one_stage(q, metric):
        return torch.topk(scores(q, metric), k, dim=1, largest=False).indices

    def two_stage(q, metric):
        s = scores(q, metric).view(q.shape[0], n // BLOCK, BLOCK)
        v, i = torch.topk(s, k, dim=2, largest=False)               # the best k of each block
        i = i + (torch.arange(n // BLOCK, device=x.device) * BLOCK)[None, :, None]
        v, i = v.reshape(q.shape[0], -1), i.reshape(q.shape[0], -1)
        best = torch.topk(v, k, dim=1, largest=False).indices
        return torch.gather(i, 1, best)

    def l1_tile(q, xs):
        d = (q[:, None, :] - xs[None, :, :]).abs().sum(dim=2)
        return torch.topk(d, k, dim=1, largest=False)

    return {
        ("l2", "one-stage top-k"): lambda q: one_stage(q, "l2"),
        ("l2", "two-stage top-k"): lambda q: two_stage(q, "l2"),
        ("ip", "one-stage top-k"): lambda q: one_stage(q, "ip"),
        ("ip", "two-stage top-k"): lambda q: two_stage(q, "ip"),
    }, l1_tile


def run_chunks(fn, q, size):
    """fn over q in chunks of a fixed size (the last one padded), so a compiled fn sees a single shape."""
    out = []
    for s in range(0, len(q), size):
        qs = q[s:s + size]
        if len(qs) < size:
            qs = torch.cat([qs, qs[:1].expand(size - len(qs), -1)])
        out.append(fn(qs)[:min(size, len(q) - s)])
    return torch.cat(out)


def run_l1(tile, x, q, k):
    """L1: each tile's best k, merged as the tiles go."""
    out = []
    n = x.shape[0]
    for s in range(0, len(q), L1_Q):
        qs = q[s:s + L1_Q]
        if len(qs) < L1_Q:
            qs = torch.cat([qs, qs[:1].expand(L1_Q - len(qs), -1)])
        bv = bi = None
        for t in range(0, n, L1_N):
            v, i = tile(qs, x[t:t + L1_N])
            i = i + t
            if bv is None:
                bv, bi = v, i
            else:
                v, i = torch.cat([bv, v], 1), torch.cat([bi, i], 1)
                top = torch.topk(v, k, dim=1, largest=False).indices
                bv, bi = torch.gather(v, 1, top), torch.gather(i, 1, top)
        out.append(bi[:min(L1_Q, len(q) - s)])
    return torch.cat(out)


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--device", default="auto", choices=("auto", "tpu", "cpu"))
    p.add_argument("--n", type=int, default=1_000_000)
    p.add_argument("--d", type=int, default=128)
    p.add_argument("--batches", default="100,1000,10000")
    p.add_argument("--l1-batch", type=int, default=100, help="L1's single batch size (its tiles are slow)")
    p.add_argument("--check-queries", type=int, default=100)
    a = p.parse_args()
    if a.n % BLOCK or a.n % L1_N and a.n > L1_N:
        raise SystemExit(f"--n must be a multiple of {BLOCK} and of {L1_N} (or below {L1_N})")
    be = Backend(a.device)
    batches = tuple(map(int, a.batches.split(",")))
    k = 10
    rng = np.random.default_rng(0)
    xb = rng.standard_normal((a.n, a.d), dtype=np.float32)
    xq = rng.standard_normal((max(batches), a.d), dtype=np.float32)
    print(f"{be.describe()}: {a.n:,} Gaussian vectors, {a.d} dimensions, k = {k}", flush=True)
    t = time.perf_counter()
    truth = {m: exact_numpy(xb, xq[:a.check_queries], k, m) for m in ("l2", "ip")}
    truth["l1"] = exact_numpy(xb, xq[:min(a.check_queries, a.l1_batch)], k, "l1")
    print(f"  NumPy's exact neighbors for {a.check_queries} queries: {time.perf_counter() - t:.1f} s", flush=True)
    x, q = torch.from_numpy(xb).to(be.device), torch.from_numpy(xq).to(be.device)
    be.sync()

    def agreement(found, ref):
        found = found[:len(ref)]
        recall = np.mean([len(set(f) & set(r)) / len(r) for f, r in zip(found, ref, strict=True)])
        return recall, np.mean([np.array_equal(f, r) for f, r in zip(found, ref, strict=True)])

    def measure(call, qb):
        call(qb)                                                      # compiles, warms up
        be.sync()
        best = float("inf")
        for _ in range(3):
            t = time.perf_counter()
            call(qb)
            be.sync()
            best = min(best, time.perf_counter() - t)
        be.reset_peak()
        with HostPeak() as h:
            call(qb)
            be.sync()
        return len(qb) / best, be.peak_mb(), h.mb

    def fmt_mem(dev, host):
        return f"{'?' if dev is None else f'{dev:,.0f}'} / {'?' if host is None else f'{host:,.0f}'}"

    print("  queries per second, then memory per call, MB: the accelerator's peak / the host's peak; "
          "recall@10 and exact matches against NumPy", flush=True)
    for precision in ("medium", "highest"):
        torch.set_float32_matmul_precision(precision)
        searches, l1_tile = make_searches(x, k)
        for compiled in (False, True):
            if compiled and be.name == "cpu":
                continue
            if compiled:
                torch._dynamo.reset()
            mode = f"precision {precision}, {'compiled' if compiled else 'eager'}"
            print(f"\n  {mode}" + "".join(f"{f'batch {b:,}':>22s}" for b in batches), flush=True)
            for (metric, how), fn in searches.items():
                fn = torch.compile(fn, backend=be.compile_backend) if compiled else fn

                def call(qb, fn=fn):
                    return run_chunks(fn, qb, CHUNK)
                label = f"{metric}, {how}"
                print(f"    {label:30s}", end="", flush=True)
                mems = []
                try:
                    found = call(q[:a.check_queries]).cpu().numpy()
                    for b in batches:
                        qps, dev, host = measure(call, q[:b])
                        print(f"{qps:22,.0f}", end="", flush=True)
                        mems.append(fmt_mem(dev, host))
                    recall, same = agreement(found, truth[metric])
                    print(f"\n    {'  memory':30s}" + "".join(f"{m:>22s}" for m in mems), flush=True)
                    print(f"      against NumPy on {len(truth[metric])} queries: recall@10 {recall:.4f}, "
                          f"exact matches {same:.4f}", flush=True)
                except Exception as e:  # noqa: BLE001 (report and go on: one failure should not hide the rest)
                    print(f"\n      failed: {type(e).__name__}: {str(e)[:300]}", flush=True)
            if precision == "medium":                                 # L1 has no matrix multiply: once per mode
                tile = torch.compile(l1_tile, backend=be.compile_backend) if compiled else l1_tile
                print(f"    {'l1, tiles':30s}", end="", flush=True)
                try:
                    qb = q[:a.l1_batch]
                    qps, dev, host = measure(lambda qq: run_l1(tile, x, qq, k), qb)
                    recall, same = agreement(run_l1(tile, x, qb, k).cpu().numpy(), truth["l1"])
                    print(f"{qps:22,.0f}  (batch {a.l1_batch}; memory {fmt_mem(dev, host)}; recall@10 {recall:.4f}, "
                          f"exact matches {same:.4f})", flush=True)
                except Exception as e:  # noqa: BLE001
                    print(f"\n      failed: {type(e).__name__}: {str(e)[:300]}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
!python -u tpu_probe.py